In [ ]:
"""
《常暗之厢》模组解析工作流。
核心逻辑：加载文档 → 解析场景/事件 → 需求匹配 → 交叉验证 → 文学性扩充。

注意：parsers.py 和 pipeline.py 已废弃，由 src/library/ + src/module_designer/layered_parser.py 替代。
旧文件移至 src/archive/。如仍需运行此工作流，请取消下面 archive 导入行的注释。
"""
import sys
import json

# 将 src/ 加入路径以导入依赖模块
sys.path.insert(0, "../src")

from utils import parser, estimate_and_truncate_context
# 以下函数已废弃，由 module_designer/layered_parser 替代
from archive.parsers import parse_scenes_from_document, parse_events_from_document
from archive.pipeline import resolve_requirements, cross_validate_and_revise, expand_scene_descriptions
from llm import call_deepseek_summarize

In [2]:
content = parser("../常暗之厢（7版规则，简体修正版）.docx")
content = estimate_and_truncate_context(content)

[Token 预估] content: 12,458 tokens
[Token 预估] 合计: 12,458 tokens (上限: 300,000)
[Token 预估] 无需截断，直接使用原文


In [4]:
res_scenes = parse_scenes_from_document(content=content)

In [5]:
res_event = parse_events_from_document(content=content)

In [ ]:
# 保存结果到 JSON 文件
with open("../data/output/scene_output.json", "w", encoding="utf-8") as f:
    json.dump(res_scenes, f, ensure_ascii=False, indent=2)

print(f"已保存至 data/output/scene_output.json，共 {len(res_scenes)} 个场景")

In [ ]:
# 保存结果到 JSON 文件
with open("../data/output/res_event.json", "w", encoding="utf-8") as f:
    json.dump(res_event, f, ensure_ascii=False, indent=2)

print(f"已保存至 data/output/res_event.json，共 {len(res_event)} 个场景")

In [ ]:
summary = call_deepseek_summarize(
    content,
    max_chars=1000,
    focus="故事整体的背景和氛围",
    output_path="../data/summary.txt"
)

In [ ]:
result = resolve_requirements(
    events_path="../data/output/res_event.json",
    scenes_path="../data/output/scene_output.json",
    content=content
    )

In [ ]:
result = cross_validate_and_revise(
    events_path="../data/output/res_event_resolved.json",
    scenes_path="../data/output/scene_output_resolved.json",
    content=content,
    auto_revise=True,
)

In [ ]:
expanded_scenes = expand_scene_descriptions(
    scenes_path="../data/output/scene_output_revised.json",
    events_path="../data/output/res_event_revised.json",
    content=content,
    output_path="../data/output/scene_output_expanded.json",
)

---

## 新工作流（三层解析 — layered_parser + layered_pipeline）

以下 Cell 演示使用新的 `module_designer` 三层解析器一键生成 L1/L2/L3 JSON。

**注意**：需要 DeepSeek API 可用。每次 LLM 调用消耗 token。

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  新三层解析工作流（一键导入模组）
# ═══════════════════════════════════════════════════════════════
from module_designer import parse_module, save_module, run_pipeline
from library import WeaponLibrary, EnemyLibrary, ContentInjector
from llm import call_deepseek

# ── 初始化库 ──
wl = WeaponLibrary()
wl.load_core()
el = EnemyLibrary()
el.load_core()
injector = ContentInjector(wl, el)

# ── 一键解析 ──
# results = parse_module(content, call_deepseek)
# save_module(results, "../data/modules/常暗之厢/")

# ── 后处理管线 ──
# pipeline_result = run_pipeline(
#     results["L1"], results["L2"], results["L3"],
#     injector=injector, weapon_lib=wl, enemy_lib=el,
# )
# save_module(
#     {"L1": pipeline_result.l1_data, "L2": pipeline_result.l2_data, "L3": pipeline_result.l3_data},
#     "../data/modules/常暗之厢/",
# )

print("新三层解析流程就绪（暂注释，取消注释即可运行）。")
print("管线流程：parse_module() → run_pipeline() → save_module()")